# Graph Analysis Query Playbook

Notebook de consultas analiticas sobre el grafo social en Neo4j.

Objetivo:
- responder preguntas de comunidades, influencia, patrones, recomendaciones y caminos
- dejar la pregunta de negocio explicitamente asociada a cada consulta
- centralizar consultas Cypher y GDS en un solo lugar ejecutable


## Alcance Metodologico

Notas importantes sobre este notebook:
- `CONOCE_A` es una relacion inferida, no una verdad operacional directa.
- `CONOCE_A` se deriva de comportamiento observado: coasistencia, coincidencias VIP observadas y reservas compartidas por evento.
- Las consultas de negocio de este notebook evitan depender de variables latentes del generador sintetico como `social_group_id`, `mixing_score`, `invite_power` o `newcomer_affinity`.
- Cuando una pregunta admite varias interpretaciones, se incluyen consultas separadas o notas explicativas.


## Requisitos

Antes de ejecutar este notebook:
- verifica que `../.env` tenga las credenciales correctas
- confirma que Neo4j este accesible por Bolt
- confirma que Graph Data Science este instalado en la instancia

Variables esperadas en `.env`:
- `NEO4J_URI`
- `NEO4J_USERNAME`
- `NEO4J_PASSWORD`
- `NEO4J_DATABASE`


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import os

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ENV_PATH = PROJECT_ROOT / '.env'
load_dotenv(ENV_PATH, override=True)

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')
if NEO4J_DATABASE in ('', 'None', 'none'):
    NEO4J_DATABASE = None

required_env = {
    'NEO4J_URI': NEO4J_URI,
    'NEO4J_USERNAME': NEO4J_USERNAME,
    'NEO4J_PASSWORD': NEO4J_PASSWORD,
}
missing_env = [key for key, value in required_env.items() if not value]
if missing_env:
    raise ValueError(f'Missing required environment variables: {missing_env}')

SESSION_KWARGS = {'database': NEO4J_DATABASE} if NEO4J_DATABASE else {}

print('PROJECT_ROOT =', PROJECT_ROOT)
print('NEO4J_URI =', NEO4J_URI)
print('NEO4J_DATABASE =', repr(NEO4J_DATABASE))


PROJECT_ROOT = c:\Users\djtor\OneDrive\Documentos\GitHub\DidierParody\deluxe-analyze
NEO4J_URI = bolt://34.28.39.13:7687
NEO4J_DATABASE = 'neo4j'


In [3]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

driver.verify_connectivity()
print('Neo4j connectivity OK')


Neo4j connectivity OK


In [4]:
def run_query(query: str, parameters: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    with driver.session(**SESSION_KWARGS) as session:
        result = session.run(query, parameters or {})
        return [dict(record) for record in result]

def run_df(query: str, parameters: dict[str, Any] | None = None) -> pd.DataFrame:
    return pd.DataFrame(run_query(query, parameters))

def run_write(query: str, parameters: dict[str, Any] | None = None) -> None:
    with driver.session(**SESSION_KWARGS) as session:
        session.run(query, parameters or {}).consume()

def graph_exists(graph_name: str) -> bool:
    rows = run_query(
        """
        CALL gds.graph.list()
        YIELD graphName
        WHERE graphName = $graph_name
        RETURN count(*) > 0 AS exists
        """,
        {'graph_name': graph_name},
    )
    return bool(rows[0]['exists']) if rows else False

def show_query(query_id: str) -> None:
    entry = QUERY_CATALOG[query_id]
    print('Question:', entry['question'])
    print('Category:', entry['category'])
    if entry.get('notes'):
        print('Notes:', entry['notes'])
    print('\nQuery:\n')
    print(entry['query'])

def run_catalog(query_id: str, parameters: dict[str, Any] | None = None) -> pd.DataFrame:
    entry = QUERY_CATALOG[query_id]
    print('Question:', entry['question'])
    print('Category:', entry['category'])
    if entry.get('notes'):
        print('Notes:', entry['notes'])
    return run_df(entry['query'], parameters)


## Validacion Base

Conviene correr estas consultas primero para validar que el grafo ya esta cargado.

In [5]:
VALIDATION_QUERIES = {
    'usuarios': "MATCH (u:Usuario) RETURN count(u) AS total",
    'eventos': "MATCH (e:Evento) RETURN count(e) AS total",
    'mesas': "MATCH (m:Mesa) RETURN count(m) AS total",
    'conoce_a': "MATCH ()-[r:CONOCE_A]-() RETURN count(r) AS total",
}

validation_summary = {
    name: run_query(query)[0]['total']
    for name, query in VALIDATION_QUERIES.items()
}
validation_summary


{'usuarios': 361, 'eventos': 35, 'mesas': 59, 'conoce_a': 83398}

## Proyeccion GDS

La red social se proyecta desde `CONOCE_A` usando `tie_strength` como peso.
Si ya existe una proyeccion llamada `socialGraph`, la eliminamos y la reconstruimos.

In [6]:
GRAPH_NAME = 'socialGraph'

if graph_exists(GRAPH_NAME):
    run_write("CALL gds.graph.drop($graph_name)", {'graph_name': GRAPH_NAME})
    print(f'Dropped existing graph: {GRAPH_NAME}')

GDS_PROJECT_QUERY = '''
CALL gds.graph.project(
  $graph_name,
  'Usuario',
  {
    CONOCE_A: {
      type: 'CONOCE_A',
      orientation: 'UNDIRECTED',
      properties: ['tie_strength']
    }
  }
)
YIELD graphName, nodeCount, relationshipCount, projectMillis
RETURN graphName, nodeCount, relationshipCount, projectMillis
'''

run_df(GDS_PROJECT_QUERY, {'graph_name': GRAPH_NAME})


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($graph_name)'


Dropped existing graph: socialGraph


,graphName,nodeCount,relationshipCount,projectMillis
0,socialGraph,361,83398,99


## Catalogo de Preguntas y Consultas

Cada entrada del catalogo tiene:
- `category`: grupo de negocio
- `question`: la pregunta que responde
- `query`: la consulta Cypher/GDS asociada
- `notes`: contexto metodologico cuando hace falta


In [7]:
QUERY_CATALOG: dict[str, dict[str, str]] = {
    'top_relaciones_sin_duplicados': {
        'category': 'Validacion',
        'question': 'Cuales son los lazos sociales inferidos mas fuertes sin duplicados espejo?',
        'notes': 'Sirve para inspeccionar la salida de CONOCE_A y validar tie_strength.',
        'query': '''
MATCH (u:Usuario)-[r:CONOCE_A]-(v:Usuario)
WHERE u.id < v.id
RETURN
  u.id AS user_1,
  v.id AS user_2,
  r.shared_events AS shared_events,
  r.shared_vip_events AS shared_vip_events,
  r.shared_reservations AS shared_reservations,
  r.tie_strength AS tie_strength
ORDER BY r.tie_strength DESC, user_1, user_2
LIMIT 20
'''
    },
    'grupos_de_clientes_louvain': {
        'category': 'Analisis de comunidades sociales',
        'question': 'Que grupos de clientes siempre asisten juntos a los mismos eventos?',
        'notes': 'Usa Louvain sobre la red inferida CONOCE_A.',
        'query': '''
CALL gds.louvain.stream('socialGraph')
YIELD nodeId, communityId
RETURN
  communityId,
  collect(gds.util.asNode(nodeId).id) AS usuarios,
  count(*) AS tamano
ORDER BY tamano DESC
'''
    },
    'subgrupos_vip_observados_louvain': {
        'category': 'Analisis de comunidades sociales',
        'question': 'Hay subgrupos dentro de los clientes VIP que nunca se mezclan con otros grupos?',
        'notes': 'VIP se define por comportamiento observado: proporcion de asistencias con ticket_tier = vip >= 0.8. No usa vip_affinity.',
        'query': '''
CALL gds.louvain.stream('socialGraph')
YIELD nodeId, communityId
WITH gds.util.asNode(nodeId) AS u, communityId
MATCH (u)-[a:ASISTIO_A]->(:Evento)
WITH u, communityId,
     avg(CASE WHEN a.ticket_tier = 'vip' THEN 1.0 ELSE 0.0 END) AS vip_ratio,
     count(a) AS total_attendances
WHERE total_attendances > 0 AND vip_ratio >= 0.8
RETURN
  communityId,
  collect(u.id) AS vip_users,
  round(avg(vip_ratio), 3) AS avg_vip_ratio,
  count(*) AS tamano
ORDER BY tamano DESC, avg_vip_ratio DESC
'''
    },
    'densidad_red_social': {
        'category': 'Analisis de comunidades sociales',
        'question': 'Que tan densamente conectada esta la red de clientes?',
        'notes': 'Cerca de 0 sugiere fragmentacion; cerca de 1 sugiere alta cohesion.',
        'query': '''
MATCH (u:Usuario)
WITH count(u) AS n
MATCH (:Usuario)-[r:CONOCE_A]-(:Usuario)
WITH n, count(r) AS m
RETURN
  n AS usuarios,
  m AS conexiones,
  round((2.0 * m) / (n * (n - 1)), 4) AS densidad
'''
    },
    'cliente_mas_influyente_pagerank': {
        'category': 'Influencia y propagacion',
        'question': 'Quien es el cliente mas influyente de la discoteca?',
        'notes': 'PageRank sobre CONOCE_A con tie_strength como peso.',
        'query': '''
CALL gds.pageRank.stream('socialGraph', {
  relationshipWeightProperty: 'tie_strength'
})
YIELD nodeId, score
RETURN
  gds.util.asNode(nodeId).id AS user_id,
  score
ORDER BY score DESC
LIMIT 20
'''
    },
    'dependencia_si_no_viene_x': {
        'category': 'Influencia y propagacion',
        'question': 'Si el cliente X no viene, que otros clientes probablemente tampoco vendran?',
        'notes': 'Aproximacion local basada en intensidad del lazo social inferido.',
        'query': '''
MATCH (u:Usuario {id: $user_id})-[r:CONOCE_A]-(v:Usuario)
RETURN
  v.id AS impacted_user,
  r.tie_strength AS intensidad,
  r.shared_events AS eventos_compartidos,
  r.shared_reservations AS reservas_compartidas
ORDER BY intensidad DESC
LIMIT 20
'''
    },
    'clientes_ancla_betweenness': {
        'category': 'Influencia y propagacion',
        'question': 'Hay clientes que funcionan como ancla de grupos grandes?',
        'notes': 'Betweenness destaca nodos puente por los que pasan muchos caminos cortos.',
        'query': '''
CALL gds.betweenness.stream('socialGraph', {
  relationshipWeightProperty: 'tie_strength'
})
YIELD nodeId, score
RETURN
  gds.util.asNode(nodeId).id AS user_id,
  score
ORDER BY score DESC
LIMIT 20
'''
    },
    'clientes_que_presentan_nuevos_acompanantes': {
        'category': 'Patrones de comportamiento en red',
        'question': 'Hay clientes que consistentemente presentan personas nuevas para ese cliente?',
        'notes': 'Mide acompanantes observados por primera vez con ese cliente, usando historial de coasistencia.',
        'query': '''
MATCH (u:Usuario)-[:ASISTIO_A]->(e:Evento)<-[:ASISTIO_A]-(v:Usuario)
WHERE u <> v
WITH u, v, min(e.event_date) AS first_seen_with_u
MATCH (u)-[:ASISTIO_A]->(ev:Evento)<-[:ASISTIO_A]-(v)
WHERE ev.event_date = first_seen_with_u
WITH u, ev.id AS event_id, count(DISTINCT v) AS nuevos_en_evento
WITH u, collect(nuevos_en_evento) AS nuevos_por_evento
CALL {
  WITH u
  MATCH (u)-[:ASISTIO_A]->(attended:Evento)
  RETURN count(DISTINCT attended) AS total_eventos
}
RETURN
  u.id AS user_id,
  total_eventos,
  size(nuevos_por_evento) AS eventos_con_nuevos_acompanantes,
  round(reduce(s = 0.0, x IN nuevos_por_evento | s + x) / toFloat(size(nuevos_por_evento)), 2) AS promedio_nuevos_por_evento,
  round(toFloat(size(nuevos_por_evento)) / toFloat(total_eventos), 3) AS proporcion_eventos_con_nuevos
ORDER BY promedio_nuevos_por_evento DESC, proporcion_eventos_con_nuevos DESC
LIMIT 20
'''
    },
    'clientes_que_traen_debutantes_al_local': {
        'category': 'Patrones de comportamiento en red',
        'question': 'Hay clientes que suelen venir acompanados por personas que estan en su primera asistencia observada al local?',
        'notes': 'Aproxima debut al local usando la primera fecha de asistencia observada del acompanante.',
        'query': '''
MATCH (u:Usuario)-[:ASISTIO_A]->(e:Evento)<-[:ASISTIO_A]-(v:Usuario)
WHERE u <> v
CALL {
  WITH v
  MATCH (v)-[:ASISTIO_A]->(ve:Evento)
  RETURN min(ve.event_date) AS first_local_date
}
WHERE e.event_date = first_local_date
WITH u, e, count(DISTINCT v) AS debutantes_en_evento
WITH u, collect(debutantes_en_evento) AS debutantes_por_evento
CALL {
  WITH u
  MATCH (u)-[:ASISTIO_A]->(attended:Evento)
  RETURN count(DISTINCT attended) AS total_eventos
}
RETURN
  u.id AS user_id,
  total_eventos,
  size(debutantes_por_evento) AS eventos_con_debutantes,
  round(reduce(s = 0.0, x IN debutantes_por_evento | s + x) / toFloat(size(debutantes_por_evento)), 2) AS promedio_debutantes_por_evento,
  round(toFloat(size(debutantes_por_evento)) / toFloat(total_eventos), 3) AS proporcion_eventos_con_debutantes
ORDER BY promedio_debutantes_por_evento DESC, proporcion_eventos_con_debutantes DESC
LIMIT 20
'''
    },
    'clientes_que_vienen_con_grupos_grandes': {
        'category': 'Patrones de comportamiento en red',
        'question': 'Hay clientes que suelen asistir acompanados por grupos grandes?',
        'notes': 'Proxy util cuando se quiere medir acompanamiento numeroso, no novedad del acompanante.',
        'query': '''
MATCH (u:Usuario)-[:ASISTIO_A]->(e:Evento)<-[:ASISTIO_A]-(v:Usuario)
WHERE u <> v
WITH u, e, count(DISTINCT v) AS companions
RETURN
  u.id AS user_id,
  count(DISTINCT e) AS eventos,
  avg(companions) AS promedio_acompanantes_por_evento,
  max(companions) AS max_acompanantes_en_un_evento
ORDER BY promedio_acompanantes_por_evento DESC, max_acompanantes_en_un_evento DESC
LIMIT 20
'''
    },
    'amigos_reservan_mismo_tipo_de_mesa': {
        'category': 'Patrones de comportamiento en red',
        'question': 'Los grupos de amigos tienden a reservar el mismo tipo de mesa evento tras evento?',
        'notes': 'Compara parejas conectadas por CONOCE_A en eventos coincidentes y mismo table_type.',
        'query': '''
MATCH (u1:Usuario)-[:CONOCE_A]-(u2:Usuario)
WHERE u1.id < u2.id
MATCH (u1)-[r1:RESERVO]->(m1:Mesa)
MATCH (u2)-[r2:RESERVO]->(m2:Mesa)
WHERE r1.event_id = r2.event_id
  AND m1.is_vip = m2.is_vip
RETURN
  u1.id AS user_1,
  u2.id AS user_2,
  m1.is_vip AS is_vip, m1.capacity AS capacity,
  count(*) AS coincidencias
ORDER BY coincidencias DESC
LIMIT 20
'''
    },
    'brokers_sociales_por_betweenness': {
        'category': 'Patrones de comportamiento en red',
        'question': 'Que clientes conectan comunidades distintas?',
        'notes': 'Betweenness es la metrica principal para brokers: identifica clientes que actuan como puentes entre clusters.',
        'query': '''
CALL gds.betweenness.stream('socialGraph', {
  relationshipWeightProperty: 'tie_strength'
})
YIELD nodeId, score
RETURN
  gds.util.asNode(nodeId).id AS user_id,
  score
ORDER BY score DESC
LIMIT 20
'''
    },
    'clientes_conectan_multiples_comunidades': {
        'category': 'Patrones de comportamiento en red',
        'question': 'Que clientes tienen vecinos en multiples comunidades detectadas?',
        'notes': 'Consulta auxiliar para interpretar mejor los brokers junto con betweenness.',
        'query': '''
CALL gds.louvain.stream('socialGraph')
YIELD nodeId, communityId
WITH collect({nodeId: nodeId, communityId: communityId}) AS assignments
UNWIND assignments AS a
WITH assignments, a.communityId AS communityId, gds.util.asNode(a.nodeId) AS u
MATCH (u)-[:CONOCE_A]-(neighbor:Usuario)
WITH u, communityId, assignments, collect(DISTINCT id(neighbor)) AS neighborIds
WITH u, communityId, [x IN assignments WHERE x.nodeId IN neighborIds | x.communityId] AS neighborCommunities
UNWIND neighborCommunities AS neighborCommunity
WITH u, communityId, collect(DISTINCT neighborCommunity) AS distinctNeighborCommunities
RETURN
  u.id AS user_id,
  communityId,
  size(distinctNeighborCommunities) AS total_communities_touched,
  size([c IN distinctNeighborCommunities WHERE c <> communityId]) AS external_communities_touched
ORDER BY external_communities_touched DESC, total_communities_touched DESC
LIMIT 20
'''
    },
    'evento_recomendado_para_cliente': {
        'category': 'Recomendaciones basadas en grafo',
        'question': 'A que evento deberia invitar a un cliente basandose en lo que asistieron sus conexiones?',
        'notes': 'Collaborative filtering simple sobre vecinos de CONOCE_A ponderados por tie_strength.',
        'query': '''
MATCH (u:Usuario {id: $user_id})-[r:CONOCE_A]-(friend:Usuario)-[:ASISTIO_A]->(e:Evento)
WHERE NOT (u)-[:ASISTIO_A]->(e)
RETURN
  e.id AS event_id,
  e.name AS event_name,
  sum(r.tie_strength) AS score_recomendacion,
  count(DISTINCT friend) AS amigos_que_fueron
ORDER BY score_recomendacion DESC, amigos_que_fueron DESC
LIMIT 10
'''
    },
    'clientes_sin_coincidir_mismo_circulo': {
        'category': 'Recomendaciones basadas en grafo',
        'question': 'Que clientes nunca han coincidido en un evento pero comparten exactamente el mismo circulo social?',
        'notes': 'Busca pares con vecinos mutuos sin coasistencia directa y sin lazo CONOCE_A directo.',
        'query': '''
MATCH (u1:Usuario)-[:CONOCE_A]-(f:Usuario)-[:CONOCE_A]-(u2:Usuario)
WHERE u1.id < u2.id
  AND NOT (u1)-[:CONOCE_A]-(u2)
  AND NOT EXISTS {
    MATCH (u1)-[:ASISTIO_A]->(:Evento)<-[:ASISTIO_A]-(u2)
  }
WITH u1, u2, count(DISTINCT f) AS mutuals
RETURN
  u1.id AS user_1,
  u2.id AS user_2,
  mutuals
ORDER BY mutuals DESC
LIMIT 20
'''
    },
    'camino_mas_corto_entre_dos_clientes': {
        'category': 'Analisis de caminos',
        'question': 'Cual es el camino mas corto entre dos clientes en la red social de la discoteca?',
        'notes': 'Usa shortestPath sobre CONOCE_A.',
        'query': '''
MATCH (source:Usuario {id: $from_user_id}), (target:Usuario {id: $to_user_id})
MATCH p = shortestPath((source)-[:CONOCE_A*]-(target))
RETURN
  [n IN nodes(p) | n.id] AS camino,
  length(p) AS distancia
'''
    },
    'clientes_aislados': {
        'category': 'Analisis de caminos',
        'question': 'Hay clientes completamente aislados sin conexiones con otros?',
        'notes': 'Identifica usuarios sin ningun lazo CONOCE_A.',
        'query': '''
MATCH (u:Usuario)
WHERE NOT (u)-[:CONOCE_A]-(:Usuario)
RETURN u.id AS user_id
ORDER BY user_id
'''
    },
    'alcance_indirecto_promocion': {
        'category': 'La pregunta mas poderosa',
        'question': 'Si lanzo una promocion para el cliente X, a cuantos clientes mas llegaria indirectamente a traves de su red social?',
        'notes': 'Mide alcance por expansion hasta 3 saltos sobre CONOCE_A.',
        'query': '''
MATCH (u:Usuario {id: $user_id})-[:CONOCE_A*1..3]-(v:Usuario)
WHERE u <> v
RETURN
  count(DISTINCT v) AS alcance_total,
  collect(DISTINCT v.id)[0..50] AS usuarios_alcanzados
'''
    }
}

pd.DataFrame([
    {
        'query_id': query_id,
        'category': entry['category'],
        'question': entry['question'],
        'notes': entry['notes'],
    }
    for query_id, entry in QUERY_CATALOG.items()
]).sort_values(['category', 'query_id']).reset_index(drop=True)


,query_id,category,question,notes
0,camino_mas_corto_entre_dos_clientes,Analisis de caminos,Cual es el camino mas corto entre dos clientes...,Usa shortestPath sobre CONOCE_A.
1,clientes_aislados,Analisis de caminos,Hay clientes completamente aislados sin conexi...,Identifica usuarios sin ningun lazo CONOCE_A.
2,densidad_red_social,Analisis de comunidades sociales,Que tan densamente conectada esta la red de cl...,Cerca de 0 sugiere fragmentacion; cerca de 1 s...
3,grupos_de_clientes_louvain,Analisis de comunidades sociales,Que grupos de clientes siempre asisten juntos ...,Usa Louvain sobre la red inferida CONOCE_A.
4,subgrupos_vip_observados_louvain,Analisis de comunidades sociales,Hay subgrupos dentro de los clientes VIP que n...,VIP se define por comportamiento observado: pr...
5,cliente_mas_influyente_pagerank,Influencia y propagacion,Quien es el cliente mas influyente de la disco...,PageRank sobre CONOCE_A con tie_strength como ...
6,clientes_ancla_betweenness,Influencia y propagacion,Hay clientes que funcionan como ancla de grupo...,Betweenness destaca nodos puente por los que p...
7,dependencia_si_no_viene_x,Influencia y propagacion,"Si el cliente X no viene, que otros clientes p...",Aproximacion local basada en intensidad del la...
8,alcance_indirecto_promocion,La pregunta mas poderosa,"Si lanzo una promocion para el cliente X, a cu...",Mide alcance por expansion hasta 3 saltos sobr...
9,amigos_reservan_mismo_tipo_de_mesa,Patrones de comportamiento en red,Los grupos de amigos tienden a reservar el mis...,Compara parejas conectadas por CONOCE_A en eve...


## Ejemplos de Ejecucion

Estas celdas muestran como correr consultas del catalogo. Cambia parametros como `user_id`, `from_user_id` o `to_user_id` segun el caso.

In [8]:
run_catalog('top_relaciones_sin_duplicados')


Question: Cuales son los lazos sociales inferidos mas fuertes sin duplicados espejo?
Category: Validacion
Notes: Sirve para inspeccionar la salida de CONOCE_A y validar tie_strength.


,user_1,user_2,shared_events,shared_vip_events,shared_reservations,tie_strength
0,csv:127,csv:29,10,10,2,16.50
1,csv:1,csv:20,10,10,1,15.75
2,csv:127,csv:135,10,10,1,15.75
3,csv:164,csv:29,10,10,1,15.75
4,csv:10,csv:46,13,3,1,15.25
5,csv:1,csv:10,9,9,2,15.00
6,csv:10,csv:164,10,10,0,15.00
7,csv:127,csv:20,10,10,0,15.00
8,csv:164,csv:324,10,10,0,15.00
9,csv:324,csv:332,10,10,0,15.00


In [9]:
run_catalog('grupos_de_clientes_louvain')


Question: Que grupos de clientes siempre asisten juntos a los mismos eventos?
Category: Analisis de comunidades sociales
Notes: Usa Louvain sobre la red inferida CONOCE_A.


,communityId,usuarios,tamano
0,251,"[csv:3, csv:4, csv:11, csv:19, csv:20, csv:23,...",103
1,23,"[csv:1, csv:8, csv:16, csv:21, csv:22, csv:24,...",97
2,27,"[csv:6, csv:9, csv:14, csv:28, csv:37, csv:39,...",67
3,220,"[csv:2, csv:5, csv:12, csv:13, csv:17, csv:18,...",51
4,242,"[csv:10, csv:15, csv:33, csv:42, csv:64, csv:9...",22
5,6,[csv:7],1
6,30,[csv:31],1
7,31,[csv:32],1
8,71,[csv:72],1
9,108,[csv:109],1


In [10]:
run_catalog('subgrupos_vip_observados_louvain')


Question: Hay subgrupos dentro de los clientes VIP que nunca se mezclan con otros grupos?
Category: Analisis de comunidades sociales
Notes: VIP se define por comportamiento observado: proporcion de asistencias con ticket_tier = vip >= 0.8. No usa vip_affinity.


,communityId,vip_users,avg_vip_ratio,tamano
0,23,"[csv:1, csv:29, csv:80, csv:87, csv:120, csv:1...",0.988,17
1,27,"[csv:28, csv:68, csv:160, csv:167, csv:175, cs...",0.989,11
2,251,"[csv:20, csv:36, csv:89, csv:101, csv:127, csv...",0.959,11
3,220,"[csv:17, csv:34, csv:150, csv:155, csv:169, cs...",0.973,9
4,242,"[csv:10, csv:93]",1.000,2


In [11]:
run_catalog('cliente_mas_influyente_pagerank')


Question: Quien es el cliente mas influyente de la discoteca?
Category: Influencia y propagacion
Notes: PageRank sobre CONOCE_A con tie_strength como peso.


,user_id,score
0,csv:46,2.383498
1,csv:164,2.360769
2,csv:328,2.351451
3,csv:10,2.260756
4,csv:324,2.169323
5,csv:356,2.130884
6,csv:354,2.124327
7,csv:127,2.103445
8,csv:1,2.080034
9,csv:87,2.069109


In [12]:
run_catalog('clientes_que_presentan_nuevos_acompanantes')


Question: Hay clientes que consistentemente presentan personas nuevas para ese cliente?
Category: Patrones de comportamiento en red
Notes: Mide acompanantes observados por primera vez con ese cliente, usando historial de coasistencia.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (u) { ... }', position=<SummaryInputPosition line=9, column=1, offset=347>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 347, 'line': 9, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (u:Usuario)-[:ASISTIO_A]->(e:Evento)<-[:ASISTIO_A]-(v:Usuario)\nWHERE u <> v\nWITH u, v, min(e.event_date) AS first_seen_with_u\nMATCH (u)-[:ASISTIO_A]->(ev:Evento)<-[:ASISTIO_A]-(v)\nWHERE ev.event_date = first_seen_with_u\nWITH u, ev.id AS event_id, count(DISTINCT v) AS nuevos_en_evento\nWITH u, collect(nuevos_en_evento) AS nuevos

,user_id,total_eventos,eventos_con_nuevos_acompanantes,promedio_nuevos_por_evento,proporcion_eventos_con_nuevos
0,csv:207,1,1,134.0,1.0
1,csv:317,1,1,115.0,1.0
2,csv:165,1,1,115.0,1.0
3,csv:170,1,1,111.0,1.0
4,csv:220,1,1,107.0,1.0
5,csv:300,1,1,107.0,1.0
6,csv:69,2,2,103.5,1.0
7,csv:296,1,1,97.0,1.0
8,csv:27,1,1,94.0,1.0
9,csv:252,1,1,92.0,1.0


In [13]:
run_catalog('brokers_sociales_por_betweenness')


Question: Que clientes conectan comunidades distintas?
Category: Patrones de comportamiento en red
Notes: Betweenness es la metrica principal para brokers: identifica clientes que actuan como puentes entre clusters.


,user_id,score
0,csv:250,341.848096
1,csv:276,335.036577
2,csv:50,330.844094
3,csv:118,324.384068
4,csv:97,322.520140
5,csv:207,314.188727
6,csv:64,303.817873
7,csv:69,303.329796
8,csv:77,297.185359
9,csv:232,295.550874


In [14]:
run_catalog('evento_recomendado_para_cliente', {'user_id': 'csv:87'})


Question: A que evento deberia invitar a un cliente basandose en lo que asistieron sus conexiones?
Category: Recomendaciones basadas en grafo
Notes: Collaborative filtering simple sobre vecinos de CONOCE_A ponderados por tie_strength.


,event_id,event_name,score_recomendacion,amigos_que_fueron
0,csv:6,Noche Deluxe 6,720.50,129
1,csv:4,Evento Especial 4,514.00,92
2,csv:11,Noche Deluxe 11,462.25,87
3,csv:7,Noche Deluxe 7,429.75,85
4,csv:20,Lanzamiento Deluxe 20,401.50,67
5,csv:14,Evento Especial 14,371.50,73
6,csv:3,Noche Deluxe 3,369.00,71
7,csv:17,Evento Especial 17,322.75,65
8,csv:21,Noche Deluxe 21,301.75,56
9,csv:1,Evento Especial 1,294.75,55


In [15]:
run_catalog('camino_mas_corto_entre_dos_clientes', {'from_user_id': 'csv:87', 'to_user_id': 'csv:333'})


Question: Cual es el camino mas corto entre dos clientes en la red social de la discoteca?
Category: Analisis de caminos
Notes: Usa shortestPath sobre CONOCE_A.


,camino,distancia
0,"[csv:87, csv:333]",1


In [16]:
run_catalog('alcance_indirecto_promocion', {'user_id': 'csv:87'})


Question: Si lanzo una promocion para el cliente X, a cuantos clientes mas llegaria indirectamente a traves de su red social?
Category: La pregunta mas poderosa
Notes: Mide alcance por expansion hasta 3 saltos sobre CONOCE_A.


,alcance_total,usuarios_alcanzados
0,339,"[csv:306, csv:290, csv:127, csv:324, csv:341, ..."


## Limpieza Opcional

Si no quieres dejar la proyeccion GDS cargada en memoria, ejecuta la siguiente celda.

In [17]:
# if graph_exists(GRAPH_NAME):
#     run_df("CALL gds.graph.drop($graph_name)", {'graph_name': GRAPH_NAME})
